# Data Preprocessing with Sklearn

Buku catatan ini akan merangkum teknik dan pemahaman pra-pemrosesan data utama. 'Sklearn' adalah perpustakaan prapemrosesan yang membentuk dasar yang kuat untuk memandu Anda melalui tugas penting ini dalam pipeline ilmu data.

## Garis Besar

* Nilai yang hilang
* Fitur polinomial
* Fitur kategoris
* Fitur numerik
* Transformasi khusus
* Penskalaan fitur
*Normalisasi

## 1. Nilai yang hilang

Pertama-tama, sebagian besar algoritme pembelajaran mesin tidak dapat menangani nilai yang hilang seperti Regresi Linier, namun, beberapa model memiliki keunggulan. Misalnya, model berbasis pohon melakukan pencarian serakah untuk meminimalkan kotoran; LightGBM memiliki opsi 'use_missing = false'; Hyperplane SVM dibangun oleh vektor pendukung.

Penting untuk mengidentifikasi nilai yang hilang dan mengetahui nilai mana yang diganti. Jawaban sederhananya adalah keputusan harus sebagian tergantung pada seberapa acak nilai yang hilang. Jika mereka benar-benar acak, mereka tidak memberikan informasi tambahan apa pun dan dapat dihilangkan atau diisi. Di sisi lain, jika tidak acak, fakta bahwa nilai hilang itu sendiri adalah informasi dan dapat dinyatakan sebagai fitur biner tambahan.

In [1]:
import numpy as np
import pandas as pd

# Example Missing Data
data = np.array([5,7,8, np.NaN, np.NaN, np.NaN, -5, 0,25,999,1,-1, np.NaN, 0, np.NaN]).reshape((5,3))
data = pd.DataFrame(data, columns = ['f1', 'f2', 'f3']) #feature 1, feature 2, feature 3
data

,f1,f2,f3
0,5.0,7.0,8.0
1,NaN,NaN,NaN
2,-5.0,0.0,25.0
3,999.0,1.0,-1.0
4,NaN,0.0,NaN


In [5]:
# check row-wise missing value 
data[data.isna().any(axis=1)]

,f1,f2,f3
1,NaN,NaN,NaN
4,NaN,0.0,NaN


### 1.1. Drop NAN

Baris atau kolom dengan banyak nilai yang hilang yang tidak berarti dapat dihapus dari data Anda dengan metode 'dropna'. Namun, itu akan menghapus nomor baris pada saat yang sama, jadi 'reset_index' akan menjadi ide yang bagus

* sumbu: 0 untuk baris, 1 untuk kolom
* thresh: jumlah non-NaN ada sehingga menjaga baris atau kolom
* inplace: perbarui bingkai

In [9]:
# drop all NA
data.dropna()

,f1,f2,f3
0,5.0,7.0,8.0
2,-5.0,0.0,25.0
3,999.0,1.0,-1.0


In [10]:
# drop 2+ NA
data.dropna(thresh=1)

,f1,f2,f3
0,5.0,7.0,8.0
2,-5.0,0.0,25.0
3,999.0,1.0,-1.0
4,NaN,0.0,NaN


In [11]:
# drop NA, reset indexing
data.dropna().reset_index(drop=True)

,f1,f2,f3
0,5.0,7.0,8.0
1,-5.0,0.0,25.0
2,999.0,1.0,-1.0


### 1.2.1. Memperhitungkan NA - Statistik

Untuk mengisi nilai yang hilang dengan strategi umum, panda menyediakan metode replace dan fillna untuk mengaitkan nilai yang hilang. Empat strategi utama adalah rata-rata, most_frequent, median dan konstanta untuk variabel kontinu, atau nilai paling sering untuk variabel kategoris

* Mudah, cepat, baik untuk kumpulan data ukuran kecil
* berarti mengubah distribusi data, menyebabkan overfiting
* hanya perhitungan berdasarkan kolom, korelasi yang hilang antar fitur
* kinerja buruk pada variabel kategoris yang dikodekan

In [13]:
# fillna by mean
data.fillna(data.mean())

,f1,f2,f3
0,5.0,7.0,8.000000
1,333.0,2.0,10.666667
2,-5.0,0.0,25.000000
3,999.0,1.0,-1.000000
4,333.0,0.0,10.666667


In [15]:
from sklearn.impute import SimpleImputer

# mean, median, most_frequent
imp_mean = SimpleImputer(strategy='mean')
imp_mean.fit_transform(data)

array([[  5.        ,   7.        ,   8.        ],
       [333.        ,   2.        ,  10.66666667],
       [ -5.        ,   0.        ,  25.        ],
       [999.        ,   1.        ,  -1.        ],
       [333.        ,   0.        ,  10.66666667]])

This pandas implementation also provides options to fill forward (ffill) or fill backward (bfill), which are convenient when working with time series.

In [12]:
# replace NA with 999 and 0 
data.replace([999.0,0], np.NaN)

,f1,f2,f3
0,5.0,7.0,8.0
1,NaN,NaN,NaN
2,-5.0,NaN,25.0
3,NaN,1.0,-1.0
4,NaN,NaN,NaN


### 1.2.2. Menghitung NA - Pemodelan

Cara populer lainnya untuk mengaitkan data yang hilang menggunakan algoritma k-nearest neighbor (KNN) untuk memprediksi nilai yang hilang. Nilai yang diisi dihitung dengan titik data k-terdekat bahwa distribusinya mirip dengan data yang hilang.

* akurasi yang lebih baik
* Pertimbangkan fitur multi-dimensi
* Melalui seluruh set pelatihan, masalah memori, komputasi berat
* sensitif dengan outlier

In [18]:
from impyute.imputation.cs import fast_knn

imputed_training = fast_knn(data, k=2)
imputed_training

,0,1,2
0,5.000000,7.000000,8.000000
1,6.987584,6.957582,8.016159
2,-5.000000,0.000000,25.000000
3,999.000000,1.000000,-1.000000
4,6.987364,0.000000,8.016157


Algoritma MICE

* fexible
* menangani data kontinu, biner, kategoris

In [20]:
from impyute.imputation.cs import mice

imputed_training = mice(data)
imputed_training

,0,1,2
0,5.000000,7.0,8.000000
1,400.911111,2.0,10.666667
2,-5.000000,0.0,25.000000
3,999.000000,1.0,-1.000000
4,604.644444,0.0,10.666667


Imputasi regresi stokastik

Imputasi regresi adalah melatih model regresi menggunakan fitur terkait untuk memprediksi nilai yang hilang, dan imputasi regresi stokastik menambahkan istilah residu acak tambahan pada algoritma di atas.

### 2. Polynomial features

**Membuat fitur polinomial adalah cara rekayasa fitur yang sederhana dan umum yang menambah kompleksitas data input numerik dengan menggabungkan fitur.**

Fitur polinomial sering dibuat ketika kita ingin memasukkan gagasan bahwa ada hubungan nonlinier antara fitur dan target. Mereka sebagian besar digunakan untuk menambah kompleksitas pada model linier dengan sedikit fitur, atau ketika kita menduga efek dari satu fitur bergantung pada fitur lain.

Jika Anda misalnya mengganti semua nilai yang hilang dengan 0, semua produk silang yang menggunakan fitur ini akan menjadi 0. Selain itu, jika Anda tidak mengganti nilai yang hilang (NaN), membuat fitur polinomial akan menimbulkan kesalahan nilai.

Sklearn menyediakan kelas 'PolynomialFeatures' untuk membuat fitur polinomial dari awal. Parameter derajat menentukan derajat maksimum polinomial. Misalnya, ketika degree diatur ke dua, fitur yang dibuat akan menjadi 1, $x_1$, $x_2$, $x^2$ dan $x_1x_2$. Parameter interaction_only memberi tahu fungsi bahwa kita hanya menginginkan fitur interaksi, yaitu 1, $x_1$, $x_2$, dan $x_1x_2$.

In [14]:
from sklearn.preprocessing import PolynomialFeatures

# Replace 999 and NaN to mean
data_clean = data.replace(999.0, np.NaN).fillna(data.mean())
print(data_clean)

poly = PolynomialFeatures(degree=3, interaction_only=True) # only create interaction terms

polynomials = pd.DataFrame(poly.fit_transform(data_clean))
polynomials

      f1   f2         f3
0    5.0  7.0   8.000000
1  333.0  2.0  10.666667
2   -5.0  0.0  25.000000
3  333.0  1.0  -1.000000
4  333.0  0.0  10.666667


,0,1,2,3,4,5,6,7
0,1.0,5.0,7.0,8.000000,35.0,40.0,56.000000,280.0
1,1.0,333.0,2.0,10.666667,666.0,3552.0,21.333333,7104.0
2,1.0,-5.0,0.0,25.000000,-0.0,-125.0,0.000000,-0.0
3,1.0,333.0,1.0,-1.000000,333.0,-333.0,-1.000000,-333.0
4,1.0,333.0,0.0,10.666667,0.0,3552.0,0.000000,0.0


Sama seperti bentuk rekayasa fitur lainnya, penting untuk membuat fitur polinomial sebelum melakukan penskalaan fitur apa pun.

## 3. Fitur kategoris

Himpunan data akan berisi variabel kategoris. Variabel ini biasanya disimpan sebagai nilai teks yang mewakili berbagai sifat. Beberapa contohnya termasuk warna ('ordinal') dan ukuran ('nominal'). Banyak algoritme pembelajaran mesin dapat mendukung nilai kategoris tanpa manipulasi lebih lanjut tetapi ada lebih banyak algoritme yang tidak, sehingga perlu untuk mengonversi fitur kategoris menjadi representasi numerik.

Sebelum Anda mulai mengubah data Anda, penting untuk mengetahui apakah fitur yang Anda kerjakan adalah ordinal (bukan nominal). Fitur ordinal paling baik digambarkan sebagai fitur dengan kategori alami dan teratur dan jarak antar kategori tidak diketahui.

In [22]:
x = np.array(['M', 'O', 'medium', 'M', 'O', 'high','F', 'A', 'high', 'F', 'AB', 'low','F', 'B', np.NaN]).reshape(5,3)

data = pd.DataFrame(x, columns = ['sex', 'blood_type', 'edu_level'])
data

,sex,blood_type,edu_level
0,M,O,medium
1,M,O,high
2,F,A,high
3,F,AB,low
4,F,B,nan


### 3.1. Satu Pengkodean Panas

**Cara paling populer untuk mengkodekan fitur kategoris adalah one-hot-encoding. Strategi dasarnya adalah mengonversi setiap nilai kategori menjadi kolom baru dan menetapkan nilai 1 atau 0 (Benar/Salah) ke kolom.** Ini memiliki manfaat karena tidak membobot nilai secara tidak benar tetapi memiliki kelemahan menambahkan lebih banyak kolom ke kumpulan data. Pada dasarnya, setiap fitur kategoris dengan n kategori diubah menjadi n fitur biner.

* Kutukan Dimensi ketika ada banyak kategori, menyebabkan model overfit
* Sparsity kehilangan kekuatan fitur, variabel kontinu lebih tinggi kepentingan fitur
* Representasi padat: memori berat
* Representasi Jarang: efisien, tetapi membutuhkan model mendukung matrik jarang (xgboost, LGBM)

In [23]:
# prefix params rename column
pd.get_dummies(data)

,sex_F,sex_M,blood_type_A,blood_type_AB,blood_type_B,blood_type_O,edu_level_high,edu_level_low,edu_level_medium,edu_level_nan
0,0,1,0,0,0,1,0,0,1,0
1,0,1,0,0,0,1,1,0,0,0
2,1,0,1,0,0,0,1,0,0,0
3,1,0,0,1,0,0,0,1,0,0
4,1,0,0,0,1,0,0,0,0,1


In [48]:
from sklearn.preprocessing import OneHotEncoder

# categories: unique value per feature, set as training set feature, or input customerized feature list
# sparse: return matrix if True, else return array
# dtype: set as np.int
# handle_unknown: set ignore for missing feature 

enc = OneHotEncoder(dtype=np.int, sparse=True, handle_unknown='ignore')
pd.DataFrame(enc.fit_transform(data).toarray(), columns=pd.get_dummies(data).columns) # enc.get_feature_names()

,sex_F,sex_M,blood_type_A,blood_type_AB,blood_type_B,blood_type_O,edu_level_high,edu_level_low,edu_level_medium,edu_level_nan
0,0,1,0,0,0,1,0,0,1,0
1,0,1,0,0,0,1,1,0,0,0
2,1,0,1,0,0,0,1,0,0,0
3,1,0,0,1,0,0,0,1,0,0
4,1,0,0,0,1,0,0,0,0,1


### 3.2. Pengkodean Numerik

Pendekatan lain untuk mengkodekan nilai kategoris adalah dengan menggunakan teknik yang disebut pengkodean angkamik. Pengkodean numerik hanyalah mengonversi setiap nilai dalam kolom menjadi angka.

Pandas menyediakan metode 'Kategoris' dan 'faktorisasi' untuk mentransbentuk variabel tipe kategoris ordinal dan nominal yang juga dapat menangani nilai yang hilang dan menghormati urutan nilai kita. Untuk kategori tidak diketahui, kita dapat membuat level baru, atau mengganti dengan kategori yang paling umum.

Dalam pustaka 'Sklearn', **'OrdinalEncoder' untuk mengonversi fitur, sedangkan 'LabelEncoder' untuk mengonversi variabel target.** OrdinalEncoder dapat menyesuaikan data yang memiliki bentuk (n_samples, n_features) sedangkan LabelEncoder hanya dapat menyesuaikan data yang memiliki bentuk (n_samples,). **Namun, kedua metode tidak mendukung kategori yang tidak dikenal, sehingga dapat menyebabkan masalah saat menerapkan model.**

* Intuitif, lurus ke depan dan dapat memberi Anda kinerja yang baik
* Nilai numerik dapat disalahartikan oleh algoritma

In [46]:
cat = pd.Categorical(data.edu_level, categories=['missing', 'low', 'medium', 'high'], ordered=True)
cat = cat.fillna('missing')
labels, unique = pd.factorize(cat, sort=True)

pd.DataFrame([cat, labels], columns=['edu_level', 'Encoding'])

,edu_level,Encoding
0,medium,2
1,high,3
2,high,3
3,low,1
4,missing,0


In [26]:
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()
pd.DataFrame(enc.fit_transform(data), columns = data.columns)

,sex,blood_type,edu_level
0,1.0,3.0,2.0
1,1.0,3.0,0.0
2,0.0,0.0,0.0
3,0.0,1.0,1.0
4,0.0,2.0,3.0


In [53]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
pd.DataFrame(enc.fit_transform(data.edu_level), columns=[data.edu_level.name])

,edu_level
0,2
1,0
2,0
3,1
4,3


### 3.3. Pengkodean Biner

Bergantung pada kumpulan data, Anda mungkin dapat menggunakan pengkodean biner untuk men-hash kardinalitas menjadi nilai biner. Ini adalah kombinasi dari pengkodean label dan satu pengkodean panas untuk membuat kolom biner yang memenuhi kebutuhan Anda untuk analisis lebih lanjut. Dalam teknik ini, pertama-tama kategori dikodekan sebagai ordinal, kemudian bilangan bulat tersebut diubah menjadi kode biner, kemudian digit dari string biner itu dibagi menjadi kolom terpisah. Ini mengkodekan data dalam dimensi yang lebih sedikit daripada satu-panas.

In [55]:
import category_encoders as ce

encoder = ce.BinaryEncoder(cols=['sex', 'blood_type', 'edu_level'])
df_binary = encoder.fit_transform(data)

df_binary

,sex_0,sex_1,blood_type_0,blood_type_1,blood_type_2,edu_level_0,edu_level_1,edu_level_2
0,0,1,0,0,1,0,0,1
1,0,1,0,0,1,0,1,0
2,1,0,0,1,0,0,1,0
3,1,0,0,1,1,0,1,1
4,1,0,1,0,0,1,0,0


### 3.4. Fitur Lain-lain

Anda mungkin menemukan kolom fitur kategoris yang menentukan rentang nilai untuk titik pengamatan, misalnya, kolom usia mungkin dijelaskan dalam bentuk kategori seperti 0-20, 20-40. Yang paling umum adalah membagi rentang ini menjadi dua kolom terpisah atau menggantinya dengan beberapa ukuran seperti rata-rata rentang itu.

In [70]:
df = pd.DataFrame({'age':['0-20', '20-40', '40-60','60-80']})
df[['start', 'end']] = pd.DataFrame(df.age.str.split('-').to_list(), index=df.index)
df

,age,start,end
0,0-20,0,20
1,20-40,20,40
2,40-60,40,60
3,60-80,60,80


## 4. Numerical features

Sama seperti data kategoris yang dapat dikodekan, fitur numerik dapat didekodekan menjadi fitur kategoris. Dua cara paling umum untuk melakukan ini adalah **diskritisasi** dan **binarisasi**.

### 4.1. Diskritisasi

Diskritisasi membagi fitur kontinu ke dalam jumlah kategori yang telah ditentukan sebelumnya. **Salah satu tujuan utama diskritisasi adalah untuk secara signifikan mengurangi jumlah interval diskrit dari atribut kontinu. Oleh karena itu, mengapa transformasi ini dapat meningkatkan kinerja model berbasis pohon.**

Sklearn menyediakan kelas 'KBinsDiscretizer' yang dapat menangani hal ini. Satu-satunya hal yang harus Anda tentukan adalah jumlah bin (n_bins) untuk setiap fitur dan cara mengkodekan bin ini (ordinal, onehot, atau onehot-dense).

Parameter strategi opsional dapat diatur ke tiga nilai:

* Seragam: Semua tempat sampah di setiap fitur memiliki lebar yang identik, tetapi sangat sensitif untuk outlier
* Kuantil (default): Semua tempat sampah di setiap fitur memiliki jumlah poin yang sama.
* kmeans: semua nilai di setiap bin memiliki pusat terdekat yang sama dari cluster k-means 1D.

In [71]:
from sklearn.preprocessing import KBinsDiscretizer

x = [[-2, 1, -4,   -1],
     [-1, 2, -3, -0.5],
     [ 0, 3, -2,  0.5],
     [ 1, 4, -1,    2]]
est = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
est.fit_transform(x)

array([[0., 0., 0., 0.],
       [1., 1., 1., 0.],
       [2., 2., 2., 1.],
       [2., 2., 2., 2.]])

### 4.2. Binarisasi

Binarisasi fitur adalah proses tresholding fitur numerik untuk mendapatkan nilai boolean. Atau dengan kata lain, tetapkan nilai boolean (Benar atau Salah) untuk setiap sampel berdasarkan ambang batas.

Secara umum binarisasi berguna sebagai teknik rekayasa fitur untuk membuat fitur baru yang menunjukkan sesuatu yang bermakna. Kelas 'Binarizer' di sklearn mengimplementasikan binarisasi dengan cara yang sangat intuitif. Satu-satunya parameter yang perlu Anda tentukan adalah ambang batas dan salinan. Semua nilai di bawah atau sama dengan ambang batas diganti dengan 0, di atasnya dengan 1.

In [13]:
from sklearn.preprocessing import Binarizer

x = [[ 1., -1.,  2.],
     [ 2.,  0.,  0.],
     [ 0.,  1., -1.]]

# binary threshold 0
transformer = Binarizer(threshold = 0)
transformer.fit_transform(x)

array([[1., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]])

## 5. Penskalaan Fitur - Standardisasi

Sebelum menerapkan transformasi penskalaan apa pun, sangat penting untuk **membagi data Anda menjadi set kereta dan set pengujian**. Jika Anda mulai menskalakan data sebelumnya, maka model Anda sudah mencari informasi dari data pengujian, yang membuat data pengujian tidak digeneralisasi ke dunia nyata dan menyebabkan overfitting. Oleh karena itu, kami tidak ingin data pengujian menjadi bagian dari pemrosesan data atau langkah pelatihan model.

**Banyak metode penskalaan ulang (standardisasi dan penskalaan min-maks) beroperasi pada fitur, namun, kita juga dapat menskalakan ulang (normalisasi) di seluruh pengamatan individu**.

### 5.1. Standardisasi

Standardisasi adalah transformasi yang memusatkan data dengan menghapus nilai rata-rata setiap fitur, dan kemudian menskalakannya dengan membagi fitur (non-konstan) dengan standar deviasinya.** Setelah menstandarkan data, rata-rata akan menjadi nol dan standar deviasi satu.

Standardisasi dapat secara drastis meningkatkan kinerja model. Misalnya, banyak elemen yang digunakan dalam fungsi objektif algoritma pembelajaran (seperti kernel RBF dari Support Vector Machines atau $L 1$ dan $L 2$ regularizer dari model linier) mengasumsikan bahwa semua fitur berpusat di sekitar nol dan memiliki varians dalam urutan yang sama. **Jika fitur memiliki varians yang orde besarnya lebih besar dari yang lain, fitur tersebut mungkin mendominasi fungsi objektif dan membuat estimator tidak dapat belajar dari fitur lain dengan benar seperti yang diharapkan.**

Misalnya, analisis komponen utama sering bekerja lebih baik menggunakan standarisasi, sedangkan penskalaan min-max sering direkomendasikan untuk jaringan saraf.

**Skala Standar**: Ini murni memusatkan data dengan menggunakan rumus berikut, di mana $\mu$ adalah rata-rata dan $\sigma$ adalah standar deviasi. Ingatlah bahwa nilai contoh keempat kita hilang, dan kita menggantinya dengan rata-rata.

std_scaler = $\frac{x-\mu}{\sigma}$

In [14]:
from sklearn.preprocessing import StandardScaler

print('Original Data')
print(data, '\n')

print('Filled Missing value')
print(data.fillna(data.mean()))

scaler = StandardScaler()
scaler.fit_transform(data.fillna(data.mean()))

Original Data
      f1   f2    f3
0    5.0  7.0   8.0
1    NaN  NaN   NaN
2   -5.0  0.0  25.0
3  999.0  1.0  -1.0
4    NaN  0.0   NaN 

Filled Missing value
      f1   f2         f3
0    5.0  7.0   8.000000
1  333.0  2.0  10.666667
2   -5.0  0.0  25.000000
3  999.0  1.0  -1.000000
4  333.0  0.0  10.666667


array([[-0.89913037,  1.91741247, -0.31933647],
       [ 0.        ,  0.        ,  0.        ],
       [-0.92654289, -0.76696499,  1.71643352],
       [ 1.82567326, -0.38348249, -1.39709705],
       [ 0.        , -0.76696499,  0.        ]])

### 5.2. MinMax Scaler

MinMaxScaler mengubah fitur dengan menskalakan setiap fitur ke rentang tertentu. Rentang ini dapat diatur dengan menentukan parameter feature_range. **Scaler ini bekerja lebih baik untuk kasus di mana distribusinya bukan Gaussian atau standar deviasi sangat kecil**. Namun, ini **sensitif terhadap outlier**, jadi jika ada outlier dalam data, Anda mungkin ingin mempertimbangkan scaler lain.

MinMax_scaler = $\frac{x-min(x)}{max(x) - min(x)}$

In [15]:
from sklearn.preprocessing import MinMaxScaler

print(data.fillna(data.mean()))
scaler = MinMaxScaler(feature_range=(-10,10))
scaler.fit_transform(data.fillna(data.mean()))

      f1   f2         f3
0    5.0  7.0   8.000000
1  333.0  2.0  10.666667
2   -5.0  0.0  25.000000
3  999.0  1.0  -1.000000
4  333.0  0.0  10.666667


array([[ -9.80079681,  10.        ,  -3.07692308],
       [ -3.26693227,  -4.28571429,  -1.02564103],
       [-10.        , -10.        ,  10.        ],
       [ 10.        ,  -7.14285714, -10.        ],
       [ -3.26693227, -10.        ,  -1.02564103]])

### 5.3. MaxAbs Scaler

MaxAbsScaler bekerja sangat mirip dengan MinMaxScaler tetapi secara otomatis menskalakan data ke rentang [-1, 1] berdasarkan **maksimum absolut**. Scaler ini dimaksudkan untuk **data yang sudah berpusat pada data nol atau jarang**. Itu tidak menggeser/memusatkan data, dan dengan demikian tidak menghancurkan jarang.

MaxAbs_scaler = $\frac{x}{Max|x|}$

In [16]:
from sklearn.preprocessing import MaxAbsScaler

print(data.fillna(data.mean()))
scaler = MaxAbsScaler()
scaler.fit_transform(data.fillna(data.mean()))

      f1   f2         f3
0    5.0  7.0   8.000000
1  333.0  2.0  10.666667
2   -5.0  0.0  25.000000
3  999.0  1.0  -1.000000
4  333.0  0.0  10.666667


array([[ 0.00500501,  1.        ,  0.32      ],
       [ 0.33333333,  0.28571429,  0.42666667],
       [-0.00500501,  0.        ,  1.        ],
       [ 1.        ,  0.14285714, -0.04      ],
       [ 0.33333333,  0.        ,  0.42666667]])

### 5.4. Scaler yang Kuat

Jika data Anda berisi banyak outlier, penskalaan menggunakan rata-rata dan standar deviasi data kemungkinan tidak akan berfungsi dengan baik. Dalam kasus ini, Anda dapat menggunakan 'RobustScaler'. **Ini menghapus median dan menskalakan data sesuai dengan rentang kuantil.**

Secara default, scaler menggunakan Inter Quartile Range (IQR), yang merupakan rentang antara kuartil ke-1 dan kuartil ke-3. Rentang kuantil dapat diatur secara manual dengan menentukan parameter quantile_range saat memulai instans baru dari 'RobustScaler'.

In [17]:
from sklearn.preprocessing import RobustScaler

print(data.fillna(data.mean()))
robust = RobustScaler(quantile_range = (0.1,0.9))
robust.fit_transform(data.fillna(data.mean()))

      f1   f2         f3
0    5.0  7.0   8.000000
1  333.0  2.0  10.666667
2   -5.0  0.0  25.000000
3  999.0  1.0  -1.000000
4  333.0  0.0  10.666667


array([[-1.02500000e+03,  6.00000000e+00, -9.25925926e+00],
       [ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00],
       [-1.05625000e+03, -1.00000000e+00,  4.97685185e+01],
       [ 2.08125000e+03,  0.00000000e+00, -4.05092593e+01],
       [ 0.00000000e+00, -1.00000000e+00,  0.00000000e+00]])

## 6. Penskalaan fitur - Normalisasi

Normalisasi adalah proses **nilai pada pengamatan individu** untuk memiliki norma satuan (jumlah panjangnya adalah 1). Jenis rescaling ini sering digunakan ketika kita memiliki banyak fitur yang setara (misalnya, klasifikasi teks ketika setiap kata atau grup n-word adalah fitur).

Dalam istilah dasar, Anda perlu menormalkan data saat algoritme memprediksi berdasarkan hubungan tertimbang yang terbentuk antara titik data. Menskalakan input ke norma unit adalah operasi umum untuk klasifikasi teks atau pengelompokan.

Meskipun ada banyak cara lain untuk menormalkan data, sklearn menyediakan tiga norma: **L1, L2 dan Max**. Saat membuat instance baru dari kelas Normalizer, Anda dapat menentukan norma yang diinginkan di bawah parameter norm.

### 6.1. Normalisasi Maks

Norma maks menggunakan maksimum absolut dan melakukan untuk sampel apa yang dilakukan MaxAbsScaler untuk fitur.

Max_normalizer = $\frac{x}{Max(x)}$

In [18]:
x = data.fillna(data.mean())
print(x)

norm_max = list(max(list(abs(i) for i in x.iloc[r])) for r in range(len(x)))

max_norm_data = pd.DataFrame(np.zeros((5,3)), columns=['f1','f2','f3'])
for i in range(len(x)):
    max_norm_data.iloc[i,] = x.iloc[i]/norm_max[i]

max_norm_data

      f1   f2         f3
0    5.0  7.0   8.000000
1  333.0  2.0  10.666667
2   -5.0  0.0  25.000000
3  999.0  1.0  -1.000000
4  333.0  0.0  10.666667


,f1,f2,f3
0,0.625,0.875000,1.000000
1,1.000,0.006006,0.032032
2,-0.200,0.000000,1.000000
3,1.000,0.001001,-0.001001
4,1.000,0.000000,0.032032


### 6.2. Norma L1

Menggunakan jumlah semua nilai sebagai dan dengan demikian memberikan penalti yang sama untuk semua parameter, memaksakan jarang.

L1_norm = $\frac{x}{sum(x)}$

In [19]:
norm_l1 = list(sum(list(abs(i) for i in x.iloc[r])) for r in range(len(x)))

norm_l1_data = pd.DataFrame(np.zeros((5,3)), columns=['f1','f2','f3'])
for i in range(len(x)):
    norm_l1_data.iloc[i,] = x.iloc[i]/norm_l1[i]
    
norm_l1_data

,f1,f2,f3
0,0.250000,0.350000,0.400000
1,0.963356,0.005786,0.030858
2,-0.166667,0.000000,0.833333
3,0.998002,0.000999,-0.000999
4,0.968962,0.000000,0.031038


### 6.3. Norma L2

Menggunakan akar kuadrat dari jumlah semua nilai kuadrat. Ini menciptakan kelancaran dan invarians rotasi. Beberapa model, seperti PCA, mengasumsikan invarians rotasi, sehingga l2 akan berkinerja lebih baik.

L2_norm = $\frac{x}{\sqrt{\sum{x_i^2}}}$

In [20]:
import math
norm_l2 = list(math.sqrt(sum(list((i**2) for i in x.iloc[r]))) for r in range(len(x)))

norm_l2_data = pd.DataFrame(np.zeros((5,3)), columns=['f1','f2','f3'])
for i in range(len(x)):
    norm_l2_data.iloc[i,] = x.iloc[i]/norm_l2[i]
    
norm_l2_data

,f1,f2,f3
0,0.425628,0.595880,0.681005
1,0.999469,0.006003,0.032015
2,-0.196116,0.000000,0.980581
3,0.999999,0.001001,-0.001001
4,0.999487,0.000000,0.032016
